# ПЗ 4 — Извлечение аудио и распознавание речи через Whisper

**Задача:** из видеофайла вытащить аудиодорожку, прогнать через Whisper (OpenAI), получить текстовый транскрипт с таймкодами.

**Стек:** `moviepy`, `openai-whisper`, `ffmpeg`

In [ ]:
!pip install moviepy openai-whisper -q
!apt-get install -y ffmpeg -q

In [ ]:
import os
import whisper
from moviepy.editor import VideoFileClip
import pandas as pd
from IPython.display import display

AUDIO_PATH  = '/content/outputs/audio.wav'
OUTPUT_DIR  = '/content/outputs/ocr_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Загрузить видео
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# Извлечь аудио
clip = VideoFileClip(video_path)
clip.audio.write_audiofile(AUDIO_PATH, verbose=False, logger=None)
print(f'Аудио сохранено: {AUDIO_PATH}')

In [ ]:
# Загрузить модель Whisper (base — быстро; medium/large — точнее)
model = whisper.load_model('base')
result = model.transcribe(AUDIO_PATH, language='ru', verbose=False)
print('Транскрипция завершена')

In [ ]:
# Полный текст
print(result['text'][:1000])

In [ ]:
# Сегменты с таймкодами
segments = []
for seg in result['segments']:
    segments.append({
        'start': round(seg['start'], 2),
        'end':   round(seg['end'],   2),
        'text':  seg['text'].strip(),
    })

df = pd.DataFrame(segments)
df.to_csv(f'{OUTPUT_DIR}/whisper_transcript.csv', index=False, encoding='utf-8-sig')
display(df.head(15))